# analyze_all_methods - all 6 approaches, detailed style

Same look as `analyze_detailed_accuracy`: **median + IQR bands** across seeds, boxplot-per-method
distributions, the 2-panel revealed-shots view, and circuit-coverage. Reads
`all_methods_comparison/` (from `compare_all_methods.ipynb`).

**LM** appears in the final-distribution / shot-efficiency / table views (it has a final fit), but not
in the per-iteration convergence / growth panels (no iteration trace).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

candidates = [
    Path.cwd(), Path.cwd() / "seed_sweep_experiments",
    Path.cwd() / "GST_POUNDERS" / "seed_sweep_experiments",
    Path("/workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments"),
]
EXPERIMENT_DIR = next((p.resolve() for p in candidates if (p / "gst_seed_experiment.py").exists()), None)
RESULTS_DIR = EXPERIMENT_DIR / "all_methods_comparison"
INF  = "mean_gate_entanglement_infidelity_to_truth"
SPAM = "mean_spam_vector_l2_error_to_truth"
TARGET = 1e-4

METHOD_ORDER = ["fixed_FPR", "no_FPR", "adaptive_D", "LM"] #, "adaptive_A" , "adaptive_L"
COLORS = {"fixed_FPR": "#E69F00", "no_FPR": "#009E73", "adaptive_D": "#D55E00",
          "adaptive_A": "#56B4E9", "adaptive_L": "#0072B2", "LM": "#555555"}
MARKERS = {"fixed_FPR": "s", "no_FPR": "^", "adaptive_D": "o",
           "adaptive_A": "D", "adaptive_L": "v", "LM": "*"}
FOLDER_TO_LABEL = {"fixed_fpr": "fixed_FPR", "fixed_no_fpr": "no_FPR", "adaptive_D": "adaptive_D",
                   "adaptive_A": "adaptive_A", "adaptive_L": "adaptive_L", "lm": "LM"}

csv = RESULTS_DIR / "all_methods_summary.csv"
if not csv.exists():
    raise FileNotFoundError(f"No results at {csv}. Run compare_all_methods.ipynb first.")
summary = pd.read_csv(csv)
summary[INF] = pd.to_numeric(summary[INF], errors="coerce")

def load_iter_table(filename):
    """Long dataframe (method=label, seed, ...) from every seed_*/<folder>/<filename>."""
    frames = []
    for path in sorted(RESULTS_DIR.glob(f"seed_*/*/{filename}")):
        folder = path.parent.name
        if folder not in FOLDER_TO_LABEL:
            continue
        try:
            frame = pd.read_csv(path)
        except Exception:
            continue
        if frame.empty:
            continue
        seed = int(path.parents[1].name.split("_")[-1])
        frame.insert(0, "method", FOLDER_TO_LABEL[folder]); frame.insert(0, "seed", seed)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def finite_series(frame, column):
    if column not in frame.columns:
        return pd.Series(dtype=float)
    v = pd.to_numeric(frame[column], errors="coerce")
    return v[np.isfinite(v)]

def method_boxplot(axis, frame, column, title, ylabel, log_scale=False, target=None):
    methods = [m for m in METHOD_ORDER if m in set(frame["method"])]
    vals, kept = [], []
    for m in methods:
        cur = finite_series(frame[frame["method"] == m], column)
        if cur.empty:
            continue
        vals.append(cur.to_numpy(float)); kept.append(m)
    if not vals:
        axis.text(0.5, 0.5, f"No {column} data", ha="center", va="center"); axis.set_axis_off(); return
    try:
        boxes = axis.boxplot(vals, tick_labels=kept, patch_artist=True, showfliers=True)
    except TypeError:
        boxes = axis.boxplot(vals, labels=kept, patch_artist=True, showfliers=True)
    for box, m in zip(boxes["boxes"], kept):
        box.set(facecolor=COLORS[m], edgecolor=COLORS[m], alpha=0.45)
    for med in boxes["medians"]:
        med.set(color="black", linewidth=1.5)
    if log_scale:
        axis.set_yscale("log")
    if target is not None:
        axis.axhline(target, color="black", linestyle="--", linewidth=1.2, label="target")
        axis.legend()
    axis.set_title(title); axis.set_ylabel(ylabel)
    axis.tick_params(axis="x", rotation=20); axis.grid(axis="y", alpha=0.25, which="both")

def method_violin(axis, frame, column, title, ylabel, log_scale=False, target=None):
    """Violin (empirical PDF) per method with the raw seed points overlaid, so tails
    are visible and honest even at small n. KDE is built in log10 space when log_scale."""
    methods = [m for m in METHOD_ORDER if m in set(frame["method"])]
    data, kept = [], []
    for m in methods:
        cur = finite_series(frame[frame["method"] == m], column)
        if log_scale:
            cur = cur[cur > 0]
        if cur.empty:
            continue
        data.append(cur.to_numpy(float)); kept.append(m)
    if not data:
        axis.text(0.5, 0.5, f"No {column} data", ha="center", va="center"); axis.set_axis_off(); return
    pos = np.arange(1, len(kept) + 1)
    plot = [np.log10(d) if log_scale else d for d in data]
    multi = [i for i, d in enumerate(plot) if len(d) >= 2]   # KDE needs >= 2 points
    if multi:
        parts = axis.violinplot([plot[i] for i in multi], positions=pos[multi],
                                showextrema=False, widths=0.8)
        for body, i in zip(parts["bodies"], multi):
            body.set(facecolor=COLORS[kept[i]], edgecolor=COLORS[kept[i]], alpha=0.30)
    rng = np.random.default_rng(0)
    for i, (d, m) in enumerate(zip(plot, kept)):
        jit = (rng.random(len(d)) - 0.5) * 0.14
        axis.scatter(pos[i] + jit, d, s=24, color=COLORS[m], edgecolor="black",
                     linewidth=0.4, zorder=3, alpha=0.9)
        axis.plot([pos[i] - 0.28, pos[i] + 0.28], [np.median(d)] * 2,
                  color="black", lw=1.7, zorder=4)   # median bar
    axis.set_xticks(pos); axis.set_xticklabels(kept, rotation=20)
    if log_scale:
        lo, hi = axis.get_ylim()
        ticks = np.arange(int(np.floor(lo)), int(np.ceil(hi)) + 1)
        axis.set_yticks(ticks); axis.set_yticklabels([f"$10^{{{int(t)}}}$" for t in ticks])
        if target is not None:
            axis.axhline(np.log10(target), color="black", ls="--", lw=1.2, label="target"); axis.legend()
    elif target is not None:
        axis.axhline(target, color="black", ls="--", lw=1.2, label="target"); axis.legend()
    axis.set_title(title); axis.set_ylabel(ylabel)
    axis.grid(axis="y", alpha=0.25)


def convergence_band(ax, iter_acc, metric, ylabel, title):
    plotted = False
    for m in [x for x in METHOD_ORDER if x in set(iter_acc["method"])]:
        part = iter_acc[iter_acc["method"] == m]
        if part.empty or metric not in part.columns:
            continue
        g = part.groupby("iteration")[metric]
        x = np.asarray(sorted(g.groups), dtype=int)
        med = g.median().reindex(x).to_numpy(float)
        q25 = g.quantile(0.25).reindex(x).to_numpy(float)
        q75 = g.quantile(0.75).reindex(x).to_numpy(float)
        ax.plot(x, med, color=COLORS[m], marker=MARKERS[m], markevery=max(1, len(x) // 12), label=m, lw=2.0)
        ax.fill_between(x, q25, q75, color=COLORS[m], alpha=0.18)
        plotted = True
    ax.set_yscale("log"); ax.axhline(TARGET, color="black", ls="--", lw=1.1, label=f"target {TARGET:.0e}")
    ax.set_xlabel("POUNDERS iteration"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.grid(alpha=0.25, which="both")
    if plotted:
        ax.legend()
    return plotted

SAVE = True
seeds = sorted(summary["seed"].unique())
print("methods:", [m for m in METHOD_ORDER if m in set(summary["method"])], "| seeds:", seeds)

## Final accuracy & cost distributions (across seeds)

Boxplot per method — median line, box = IQR. Includes **LM**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
method_violin(axes[0], summary, INF, "Final mean gate infidelity to truth",
              "entanglement infidelity (violin + seed points)", log_scale=True, target=TARGET)
method_boxplot(axes[1], summary, "accounted_revealed_shots", "Accounted revealed shots (budget)",
               "accounted revealed shots", log_scale=True)
fig.tight_layout()
if SAVE:
    fig.savefig(RESULTS_DIR / "all_methods_final_distributions.png", dpi=200, bbox_inches="tight")
plt.show()

## Infidelity convergence (median + IQR across seeds)

LM omitted (no per-iteration trace).

In [ ]:
iter_acc = load_iter_table("iteration_accuracy.csv")
if iter_acc.empty:
    print("No iteration_accuracy.csv under", RESULTS_DIR, "- run compare_all_methods first.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    convergence_band(ax, iter_acc, INF, "mean gate infidelity to truth (log)",
                     "GST convergence to truth across seeds")
    fig.tight_layout()
    if SAVE:
        fig.savefig(RESULTS_DIR / "all_methods_convergence.png", dpi=200, bbox_inches="tight")
    plt.show()

## Convergence incl. LM (normalized progress)

Overlays **LM** with the POUNDERS methods. Because LM's progress is indexed by **max-length stage** (L=1..64) and POUNDERS by trust-region **iteration**, the shared x-axis is **normalized optimization progress** (0 = start → 1 = converged) — a fair shape comparison; the endpoints are the exact final infidelities. LM shown as its per-max-length-stage descent.

In [ ]:
# ==== Convergence incl. LM, normalized progress (0=start -> 1=converged) ====
def _load_lm_traj():
    frames = []
    for path in sorted(RESULTS_DIR.glob("seed_*/lm/lm_trajectory.csv")):
        try:
            fr = pd.read_csv(path)
        except Exception:
            continue
        if fr.empty:
            continue
        fr.insert(0, "seed", int(path.parents[1].name.split("_")[-1]))
        frames.append(fr)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _band(g):
    x = np.asarray(sorted(g.groups), dtype=float)
    return (x, g.median().reindex(x).to_numpy(float),
            g.quantile(0.25).reindex(x).to_numpy(float),
            g.quantile(0.75).reindex(x).to_numpy(float))

def _norm(x):
    span = max(x.max() - x.min(), 1.0)
    return (x - x.min()) / span

lm_traj = _load_lm_traj()
if iter_acc.empty:
    print("No POUNDERS iteration data - run compare_all_methods first.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    for m in [x for x in METHOD_ORDER if x in set(iter_acc["method"])]:
        part = iter_acc[iter_acc["method"] == m]
        if part.empty or INF not in part.columns:
            continue
        x, med, q25, q75 = _band(part.groupby("iteration")[INF]); xn = _norm(x)
        ax.plot(xn, med, color=COLORS[m], marker=MARKERS[m], markevery=max(1, len(xn) // 12), label=m, lw=2.0)
        ax.fill_between(xn, q25, q75, color=COLORS[m], alpha=0.18)
    if not lm_traj.empty and INF in lm_traj.columns:
        x, med, q25, q75 = _band(lm_traj.groupby("stage")[INF]); xn = _norm(x)
        ax.plot(xn, med, color=COLORS["LM"], marker=MARKERS["LM"], ms=12,
                label="LM (per max-length stage)", lw=2.4)
        ax.fill_between(xn, q25, q75, color=COLORS["LM"], alpha=0.15)
    else:
        print("No lm_trajectory.csv found - re-run LM (FORCE=True) to generate it.")
    ax.set_yscale("log"); ax.axhline(TARGET, ls="--", color="black", lw=1.1, label=f"target {TARGET:.0e}")
    ax.set_xlabel("Iterations (normalized progress)")
    ax.set_ylabel("mean gate infidelity to truth (log)")
    ax.set_title("Convergence plot")
    ax.grid(alpha=0.25, which="both"); ax.legend()
    fig.tight_layout()
    if SAVE:
        fig.savefig(RESULTS_DIR / "all_methods_convergence_with_lm.png", dpi=200, bbox_inches="tight")
    plt.show()

## SPAM convergence incl. LM (normalized progress)

Median + IQR across seeds; **LM** overlaid via its per-max-length-stage SPAM error, on the same normalized-progress axis as the infidelity version.

In [ ]:
# ==== SPAM convergence incl. LM, normalized progress (0=start -> 1=converged) ====
def _lmtraj():
    frames = []
    for path in sorted(RESULTS_DIR.glob("seed_*/lm/lm_trajectory.csv")):
        try:
            fr = pd.read_csv(path)
        except Exception:
            continue
        if fr.empty:
            continue
        fr.insert(0, "seed", int(path.parents[1].name.split("_")[-1])); frames.append(fr)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _nb(g):
    x = np.asarray(sorted(g.groups), dtype=float)
    return (x, g.median().reindex(x).to_numpy(float),
            g.quantile(0.25).reindex(x).to_numpy(float),
            g.quantile(0.75).reindex(x).to_numpy(float))

def _nx(x):
    return (x - x.min()) / max(x.max() - x.min(), 1.0)

_lm = _lmtraj()
if iter_acc.empty or SPAM not in iter_acc.columns:
    print("No SPAM data in iteration_accuracy.csv.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    for m in [x for x in METHOD_ORDER if x in set(iter_acc["method"])]:
        part = iter_acc[iter_acc["method"] == m]
        if part.empty or SPAM not in part.columns:
            continue
        x, med, q25, q75 = _nb(part.groupby("iteration")[SPAM]); xn = _nx(x)
        ax.plot(xn, med, color=COLORS[m], marker=MARKERS[m], markevery=max(1, len(xn) // 12), label=m, lw=2.0)
        ax.fill_between(xn, q25, q75, color=COLORS[m], alpha=0.18)
    if not _lm.empty and SPAM in _lm.columns:
        x, med, q25, q75 = _nb(_lm.groupby("stage")[SPAM]); xn = _nx(x)
        ax.plot(xn, med, color=COLORS["LM"], marker=MARKERS["LM"], ms=12, label="LM (per max-length stage)", lw=2.4)
        ax.fill_between(xn, q25, q75, color=COLORS["LM"], alpha=0.15)
    else:
        print("No lm_trajectory.csv - re-run LM to include it in SPAM convergence.")
    ax.set_yscale("log")
    ax.set_xlabel("normalized optimization progress (0=start -> 1=converged)")
    ax.set_ylabel("mean SPAM vector L2 error to truth (log)")
    ax.set_title("SPAM convergence incl. LM (normalized progress)")
    ax.grid(alpha=0.25, which="both"); ax.legend()
    fig.tight_layout()
    if SAVE:
        fig.savefig(RESULTS_DIR / "all_methods_spam_convergence.png", dpi=200, bbox_inches="tight")
    plt.show()

## Revealed-shot growth

Left: cumulative revealed shots vs iteration (median + IQR across seeds). Right: final accounted budget per method. Fixed methods reveal up front (flat); LM omitted from the trace.

In [ ]:
def load_shot_progress():
    frames = []
    for run_dir in sorted(RESULTS_DIR.glob("seed_*/*")):
        if not run_dir.is_dir():
            continue
        folder = run_dir.name
        if folder not in FOLDER_TO_LABEL:
            continue
        seed = int(run_dir.parent.name.split("_")[-1])
        if folder.startswith("adaptive_"):
            pth = run_dir / "adaptive_shot_events.csv"
            if not pth.exists() or pth.stat().st_size == 0:
                continue
            fr = pd.read_csv(pth)
            if fr.empty or "accounted_revealed_shots" not in fr:
                continue
            fr = fr[["iteration", "accounted_revealed_shots"]].rename(
                columns={"accounted_revealed_shots": "revealed_shots"})
        elif folder in ("fixed_fpr", "fixed_no_fpr"):
            pth = run_dir / "optimizer_progress.csv"
            if not pth.exists() or pth.stat().st_size == 0:
                continue
            fr = pd.read_csv(pth)
            if fr.empty or "cumulative_shots_revealed" not in fr:
                continue
            fr = fr.groupby("nf", as_index=False)["cumulative_shots_revealed"].last().rename(
                columns={"nf": "iteration", "cumulative_shots_revealed": "revealed_shots"})
        else:
            continue
        fr.insert(0, "method", FOLDER_TO_LABEL[folder]); fr.insert(0, "seed", seed)
        frames.append(fr)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

sp = load_shot_progress()
fig, (ax_trace, ax_final) = plt.subplots(1, 2, figsize=(13, 5.4))
if sp.empty:
    ax_trace.text(0.5, 0.5, "No shot-progress data", ha="center", va="center"); ax_trace.set_axis_off()
else:
    for m in [x for x in METHOD_ORDER if x in set(sp["method"])]:
        part = sp[sp["method"] == m]
        pivot = part.pivot_table(index="iteration", columns="seed", values="revealed_shots",
                                 aggfunc="last").sort_index()
        it = np.arange(int(pivot.index.min()), int(pivot.index.max()) + 1)
        pivot = pivot.reindex(it).ffill()
        med = pivot.median(axis=1).to_numpy(float)
        q25 = pivot.quantile(0.25, axis=1).to_numpy(float)
        q75 = pivot.quantile(0.75, axis=1).to_numpy(float)
        ax_trace.plot(it, med, color=COLORS[m], marker=MARKERS[m], markevery=max(1, len(it) // 12), label=m, lw=2.0)
        ax_trace.fill_between(it, q25, q75, color=COLORS[m], alpha=0.18)
    ax_trace.set_xlabel("POUNDERS iteration"); ax_trace.set_ylabel("cumulative revealed shots")
    _lm_shots = pd.to_numeric(summary[summary["method"] == "LM"]["accounted_revealed_shots"], errors="coerce").median()
    if np.isfinite(_lm_shots):
        ax_trace.axhline(_lm_shots, ls=":", color=COLORS["LM"], lw=2.2, label="LM (all upfront)")
    ax_trace.set_title("Shots revealed during optimization"); ax_trace.grid(alpha=0.25); ax_trace.legend()

method_boxplot(ax_final, summary, "accounted_revealed_shots",
               "Final shot budget across seeds", "accounted revealed shots")
fig.tight_layout()
if SAVE:
    fig.savefig(RESULTS_DIR / "all_methods_revealed_shots.png", dpi=200, bbox_inches="tight")
plt.show()

## Infidelity versus revealed shots (shot efficiency)

Lower-left is best. Includes LM.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for m in [x for x in METHOD_ORDER if x in set(summary["method"])]:
    g = summary[summary["method"] == m]
    ax.scatter(pd.to_numeric(g["accounted_revealed_shots"], errors="coerce"), g[INF],
               color=COLORS[m], marker=MARKERS[m], s=110, edgecolor="black", linewidth=0.5, label=m)
ax.set_xscale("log"); ax.set_yscale("log")
ax.axhline(TARGET, ls="--", color="black", lw=1.1, label=f"target {TARGET:.0e}")
ax.set_xlabel("accounted revealed shots (log)"); ax.set_ylabel("mean gate infidelity to truth (log)")
ax.set_title("Shot efficiency - all methods"); ax.grid(alpha=0.25, which="both"); ax.legend()
fig.tight_layout()
if SAVE:
    fig.savefig(RESULTS_DIR / "all_methods_shot_efficiency.png", dpi=200, bbox_inches="tight")
plt.show()

## Infidelity vs cumulative revealed shots

The shot-efficiency *trajectory* (like `analyze_detailed_accuracy`): each incumbent accuracy measurement paired with the cumulative shots revealed by that point. Line = median across seeds, band = IQR of infidelity. Hollow marker = every iteration; filled marker = the median revealed-shot total increased there. **LM** is a batch fit (all shots up front), so it appears as a single point at its total budget. Lower-left is best.


In [ ]:
# ==== Infidelity vs cumulative revealed shots (median + IQR), analyze_detailed style ====
from matplotlib.lines import Line2D

sp = load_shot_progress()
acc = load_iter_table("iteration_accuracy.csv")
if sp.empty or acc.empty:
    print("Need iteration_accuracy.csv + shot-progress data - run compare_all_methods first.")
else:
    # pair each incumbent infidelity with the cumulative revealed shots at that iteration
    eff = []
    for (seed, method), a in acc.groupby(["seed", "method"]):
        s = sp[(sp["seed"] == seed) & (sp["method"] == method)]
        if s.empty or INF not in a.columns:
            continue
        a = (a[["iteration", INF]].drop_duplicates("iteration", keep="last")
             .sort_values("iteration").set_index("iteration"))
        rev = (s.groupby("iteration")["revealed_shots"].last().sort_index()
               .reindex(a.index).ffill().bfill())
        eff.append(a.assign(revealed_shots=rev, seed=seed, method=method).reset_index())
    eff = pd.concat(eff, ignore_index=True) if eff else pd.DataFrame()

    fig, ax = plt.subplots(figsize=(9.5, 6))
    for m in [x for x in METHOD_ORDER if x in set(eff["method"])]:
        part = eff[eff["method"] == m].copy()
        part = part[np.isfinite(part["revealed_shots"]) & np.isfinite(part[INF])
                    & (part["revealed_shots"] > 0) & (part[INF] > 0)]
        if part.empty:
            continue
        it = np.arange(int(part["iteration"].min()), int(part["iteration"].max()) + 1)
        shots_by = part.pivot_table(index="iteration", columns="seed",
                                    values="revealed_shots", aggfunc="last").reindex(it).ffill()
        inf_by = part.pivot_table(index="iteration", columns="seed",
                                  values=INF, aggfunc="last").reindex(it).ffill()
        cs = shots_by.columns.intersection(inf_by.columns)
        shots_by, inf_by = shots_by[cs], inf_by[cs]
        if shots_by.shape[1] == 0:
            continue
        msh = shots_by.median(axis=1).to_numpy(float)
        minf = inf_by.median(axis=1).to_numpy(float)
        q25 = inf_by.quantile(0.25, axis=1).to_numpy(float)
        q75 = inf_by.quantile(0.75, axis=1).to_numpy(float)
        ax.plot(msh, minf, color=COLORS[m], lw=1.8, label=m)
        ax.scatter(msh, minf, s=26, marker=MARKERS[m], facecolors="none",
                   edgecolors=COLORS[m], linewidths=1.0, zorder=3)
        added = np.r_[True, np.diff(msh) > 0]
        ax.scatter(msh[added], minf[added], s=26, marker=MARKERS[m], color=COLORS[m],
                   edgecolors=COLORS[m], linewidths=1.0, zorder=4)
        ax.fill_between(msh, q25, q75, color=COLORS[m], alpha=0.18)

    # LM: batch fit (all shots up front) -> single reference point at its total budget
    lm = summary[summary["method"] == "LM"]
    if not lm.empty:
        lx = pd.to_numeric(lm["accounted_revealed_shots"], errors="coerce").median()
        ly = pd.to_numeric(lm[INF], errors="coerce").median()
        if np.isfinite(lx) and np.isfinite(ly):
            ax.scatter([lx], [ly], color=COLORS["LM"], marker=MARKERS["LM"], s=200,
                       edgecolor="black", linewidth=0.6, zorder=5, label="LM (all shots upfront)")

    ax.axhline(TARGET, color="black", ls="--", lw=1.1, label=f"target {TARGET:.0e}")
    ax.set_yscale("log")
    ax.set_xlabel("cumulative revealed shots")
    ax.set_ylabel("mean gate infidelity to truth (log)")
    ax.set_title("Infidelity vs cumulative revealed shots (median + IQR across seeds)")
    ax.grid(alpha=0.25, which="both")
    method_legend = ax.legend(loc="lower left", fontsize=9, title="method")
    ax.add_artist(method_legend)
    marks = [Line2D([0], [0], marker="o", linestyle="none", markersize=5,
                    markerfacecolor="white", markeredgecolor="black", label="all iterations"),
             Line2D([0], [0], marker="o", linestyle="none", markersize=5,
                    markerfacecolor="black", markeredgecolor="black", label="revealed shots increased")]
    ax.legend(handles=marks, loc="upper right", fontsize=9, title="marker")
    fig.tight_layout()
    if SAVE:
        fig.savefig(RESULTS_DIR / "all_methods_infidelity_vs_shots.png", dpi=200, bbox_inches="tight")
    plt.show()


## Circuit-sampling coverage (adaptive methods)

One seed (`COVERAGE_SEED`): sorted shots/circuit + Lorenz concentration. Fixed/LM omitted (uniform).

In [ ]:
COVERAGE_SEED = seeds[0]
sd = RESULTS_DIR / f"seed_{COVERAGE_SEED:06d}"
cov = []
for folder in [f for f in ["adaptive_D", "adaptive_A", "adaptive_L"] if FOLDER_TO_LABEL[f] in METHOD_ORDER]:
    npy = sd / folder / "final_shots_per_circuit.npy"
    if not npy.exists():
        continue
    summ_p = sd / folder / "summary.json"
    summ = json.loads(summ_p.read_text()) if summ_p.exists() else {}
    cov.append({"label": FOLDER_TO_LABEL[folder], "shots": np.load(npy).astype(float), "summary": summ})
if not cov:
    print("No adaptive final_shots_per_circuit.npy for seed", COVERAGE_SEED)
else:
    n = len(cov)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.0 * n), squeeze=False)
    for i, r in enumerate(cov):
        s = r["shots"]; total = s.size; base = int(s.min()); samp = int((s > base).sum())
        srt = np.sort(s)[::-1]; bud = r["summary"].get("accounted_revealed_shots", int(s.sum()))
        axL = axes[i][0]
        axL.fill_between(np.arange(total), srt, base, where=srt > base, color=COLORS.get(r["label"], "#0072B2"), alpha=0.5, step="pre")
        axL.plot(np.arange(total), srt, color=COLORS.get(r["label"], "#0072B2"), lw=1)
        axL.axhline(base, ls="--", color="gray", lw=1, label=f"baseline {base}")
        axL.set_yscale("log"); axL.set_xlabel("circuit (sorted by shots)"); axL.set_ylabel("shots (log)")
        axL.set_title(f"seed {COVERAGE_SEED} / {r['label']}: {samp}/{total} sampled ({100*samp/total:.0f}%)")
        axL.legend(fontsize=9)
        axR = axes[i][1]; cum = np.cumsum(srt) / srt.sum(); fr = np.arange(1, total + 1) / total
        axR.plot(fr, cum, color="#D55E00", label="actual"); axR.plot([0, 1], [0, 1], ls=":", color="gray", label="uniform")
        top5 = cum[max(0, int(0.05 * total) - 1)]
        axR.set_xlabel("fraction of circuits (most-sampled first)"); axR.set_ylabel("fraction of shots")
        axR.set_title(f"top 5% hold {100*top5:.0f}% of shots | total {int(s.sum()):,} / budget ~{int(bud):,}")
        axR.legend(fontsize=9); axR.grid(alpha=0.25)
    fig.tight_layout()
    if SAVE:
        fig.savefig(RESULTS_DIR / "all_methods_circuit_coverage.png", dpi=150)
    plt.show()

## Shot distribution across selected circuits (all methods)

How each method spreads its budget over the circuits it actually measures (its **selected/revealed** set = union of FPR-selected circuits; all circuits for no_FPR/LM). Median + IQR bands across seeds.

- **Left - allocation:** shots per circuit (log), circuits sorted most-sampled first, x normalized to *fraction of selected circuits* so methods with different circuit counts are comparable.
- **Right - concentration (Lorenz):** cumulative share of shots vs fraction of circuits; the diagonal is perfectly uniform. **Gini** in the legend (0 = uniform, higher = more concentrated).


In [ ]:
# ==== Shot distribution across selected circuits, all methods (median + IQR) ====
import ast
from collections import defaultdict

def _selected_shots(run_dir, folder):
    """(shots on selected circuits, accounted) for one run.
    selected = union of selected_circuit_indices across FPR calls;
    all circuits when there is no FPR history; LM reconstructed as flat mean."""
    summ_p = run_dir / "summary.json"
    if not summ_p.exists():
        return None, None
    summ = json.loads(summ_p.read_text())
    total = int(summ.get("total_circuits", 1918))
    acc = int(summ.get("accounted_revealed_shots", 0))
    if folder == "lm":
        mean = float(summ.get("mean_shots_per_circuit", acc / max(total, 1)))
        return np.full(total, mean), acc
    npy_p = run_dir / "final_shots_per_circuit.npy"
    if not npy_p.exists():
        return None, None
    shots = np.load(npy_p).astype(float)
    sel = None
    fpr = run_dir / "fpr_selection_history.csv"
    if fpr.exists() and fpr.stat().st_size > 5:
        try:
            fr = pd.read_csv(fpr, usecols=["selected_circuit_indices"])
            idx = set()
            for s in fr["selected_circuit_indices"].dropna():
                s = str(s).strip()
                if s and s != "[]":
                    idx.update(ast.literal_eval(s))
            if idx:
                sel = np.fromiter(sorted(idx), dtype=int)
        except Exception:
            sel = None
    if sel is None:
        sel = np.arange(len(shots))
    return shots[sel], acc

def _gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = x.size
    if n == 0 or x.sum() == 0:
        return 0.0
    return (2.0 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1.0) / n

GRID = np.linspace(0.0, 1.0, 200)
rank_curves = defaultdict(list)     # label -> list of shots-at-GRID-fraction arrays
lorenz_curves = defaultdict(list)   # label -> list of cumulative-share arrays
ginis = defaultdict(list)
nsel = defaultdict(list)

for seed in seeds:
    sdir = RESULTS_DIR / f"seed_{seed:06d}"
    for folder, label in FOLDER_TO_LABEL.items():
        shots, _acc = _selected_shots(sdir / folder, folder)
        if shots is None or shots.size == 0:
            continue
        srt = np.sort(shots)[::-1]                       # most-sampled first
        n = srt.size
        xf = (np.arange(1, n + 1) - 0.5) / n
        rank_curves[label].append(np.interp(GRID, xf, srt))
        xl = np.concatenate([[0.0], np.arange(1, n + 1) / n])
        yl = np.concatenate([[0.0], np.cumsum(srt) / srt.sum()])
        lorenz_curves[label].append(np.interp(GRID, xl, yl))
        ginis[label].append(_gini(shots))
        nsel[label].append(n)

def _band(curves):
    M = np.vstack(curves)
    return np.median(M, 0), np.quantile(M, 0.25, 0), np.quantile(M, 0.75, 0)

if not rank_curves:
    print("No final_shots_per_circuit data found - run compare_all_methods first.")
else:
    fig, (axA, axB) = plt.subplots(1, 2, figsize=(14.5, 6.2))
    order = [m for m in METHOD_ORDER if m in rank_curves]

    # Panel A: allocation (shots per circuit, log-y)
    for m in order:
        med, q25, q75 = _band(rank_curves[m])
        nlab = int(np.median(nsel[m]))
        axA.plot(GRID, med, color=COLORS[m], marker=MARKERS[m], markevery=20,
                 ms=8, lw=2.0, label=f"{m}  (~{nlab} circ)")
        axA.fill_between(GRID, q25, q75, color=COLORS[m], alpha=0.15)
    axA.set_yscale("log")
    axA.set_xlabel("fraction of selected circuits (most-sampled first)")
    axA.set_ylabel("shots on circuit (log)")
    axA.set_title("Shot allocation across selected circuits")
    axA.grid(alpha=0.25, which="both"); axA.legend(fontsize=9, loc="upper right")

    # Panel B: concentration (Lorenz), Gini in legend
    for m in order:
        med, q25, q75 = _band(lorenz_curves[m])
        axB.plot(GRID, med, color=COLORS[m], marker=MARKERS[m], markevery=20,
                 ms=8, lw=2.0, label=f"{m}  (G={np.median(ginis[m]):.2f})")
        axB.fill_between(GRID, q25, q75, color=COLORS[m], alpha=0.12)
    axB.plot([0, 1], [0, 1], ls=":", color="gray", lw=1.5, label="uniform (G=0)")
    axB.set_xlim(0, 1); axB.set_ylim(0, 1.005)
    axB.set_xlabel("fraction of selected circuits (most-sampled first)")
    axB.set_ylabel("cumulative fraction of shots")
    axB.set_title("Shot concentration (Lorenz curve)")
    axB.grid(alpha=0.25); axB.legend(fontsize=9, loc="lower right")

    fig.suptitle("Shot distribution across selected circuits - all methods (median +/- IQR across seeds)",
                 y=1.02, fontsize=13)
    fig.tight_layout()
    if SAVE:
        fig.savefig(RESULTS_DIR / "all_methods_shot_distribution.png", dpi=200, bbox_inches="tight")
    plt.show()

    print("median concentration across seeds:")
    print(f"  {'method':11} {'n_sel':>6} {'gini':>6} {'top1%share':>11}")
    for m in order:
        top1 = np.median([np.sort(rc)[::-1][:max(1, rc.size // 100)].sum() / rc.sum()
                          for rc in rank_curves[m]])
        print(f"  {m:11} {int(np.median(nsel[m])):>6} {np.median(ginis[m]):>6.2f} {top1:>10.1%}")


## Shot allocation per circuit index (per seed)

The allocation vs the **actual circuit index** (design order, so low index ≈ short circuits, high index ≈ long circuits) — **one panel per seed**. Set `SEEDS_TO_SHOW` to a subset (e.g. `[101]` or `seeds[:4]`) if the grid gets too tall. Each point is one selected circuit; a method's missing indices are circuits it didn't select. adaptive_D's tall spikes reveal which circuits it pours the budget into (they cluster at the long, high-index circuits); fixed_FPR / no_FPR / LM are flat bands.


In [ ]:
# ==== Shot allocation per circuit index, one panel per seed ====
import ast
from matplotlib.lines import Line2D

SEEDS_TO_SHOW = list(seeds)          # <- set to e.g. [101] or seeds[:4] to subset
_label_to_folder = {v: k for k, v in FOLDER_TO_LABEL.items()}

def _shots_by_index(run_dir, folder):
    """(indices, shots) on the method's selected circuits (union of FPR-selected
    indices; all circuits when there is no FPR history; LM reconstructed flat)."""
    summ_p = run_dir / "summary.json"
    if not summ_p.exists():
        return None, None
    summ = json.loads(summ_p.read_text())
    total = int(summ.get("total_circuits", 1918))
    if folder == "lm":
        mean = float(summ.get("mean_shots_per_circuit",
                              summ.get("accounted_revealed_shots", 0) / max(total, 1)))
        return np.arange(total), np.full(total, mean)
    npy_p = run_dir / "final_shots_per_circuit.npy"
    if not npy_p.exists():
        return None, None
    shots = np.load(npy_p).astype(float)
    sel = None
    fpr = run_dir / "fpr_selection_history.csv"
    if fpr.exists() and fpr.stat().st_size > 5:
        try:
            fr = pd.read_csv(fpr, usecols=["selected_circuit_indices"])
            idx = set()
            for s in fr["selected_circuit_indices"].dropna():
                s = str(s).strip()
                if s and s != "[]":
                    idx.update(ast.literal_eval(s))
            if idx:
                sel = np.fromiter(sorted(idx), dtype=int)
        except Exception:
            sel = None
    if sel is None:
        sel = np.arange(len(shots))
    return sel, shots[sel]

_seeds_show = list(SEEDS_TO_SHOW)
if not _seeds_show:
    print("SEEDS_TO_SHOW is empty.")
else:
    n = len(_seeds_show)
    fig, axes = plt.subplots(n, 1, figsize=(14, 3.3 * n), squeeze=False,
                             sharex=True, sharey=True)
    plotted_methods, first_ax = [], None
    for r, seed in enumerate(_seeds_show):
        ax = axes[r][0]
        sdir = RESULTS_DIR / f"seed_{int(seed):06d}"
        any_here = False
        for m in [x for x in METHOD_ORDER if x in _label_to_folder]:
            folder = _label_to_folder[m]
            idx, sh = _shots_by_index(sdir / folder, folder)
            if idx is None or idx.size == 0:
                continue
            ax.scatter(idx, sh, s=12, marker=MARKERS[m], color=COLORS[m],
                       alpha=0.55, edgecolors="none")
            any_here = True
            if m not in plotted_methods:
                plotted_methods.append(m)
        if not any_here:
            ax.text(0.5, 0.5, f"seed {int(seed)}: no data yet", ha="center", va="center",
                    transform=ax.transAxes)
            continue
        ax.set_yscale("log")
        ax.set_ylabel("shots (log)")
        ax.set_title(f"seed {int(seed)}", loc="left", fontsize=11)
        ax.grid(alpha=0.25, which="both")
        if first_ax is None:
            first_ax = ax
    axes[-1][0].set_xlabel("circuit index (design order: low \u2248 short circuits, high \u2248 long)")
    if first_ax is not None and plotted_methods:
        handles = [Line2D([0], [0], marker=MARKERS[m], linestyle="none",
                          markerfacecolor=COLORS[m], markeredgecolor="none", markersize=9, label=m)
                   for m in METHOD_ORDER if m in plotted_methods]
        first_ax.legend(handles=handles, fontsize=9, loc="upper left", ncol=len(handles))
    fig.suptitle("Shot allocation per circuit index - all seeds", y=1.002, fontsize=13)
    fig.tight_layout()
    if SAVE:
        fig.savefig(RESULTS_DIR / "all_methods_shot_allocation_by_index.png",
                    dpi=170, bbox_inches="tight")
    plt.show()


## Median summary table

In [ ]:
tbl = summary.groupby("method").agg(
    median_infidelity=(INF, "median"),
    median_accounted_shots=("accounted_revealed_shots", "median"),
    median_max_shots_per_circuit=("max_shots_per_circuit", "median"),
    n_seeds=("seed", "count"),
).reindex([m for m in METHOD_ORDER if m in set(summary["method"])])
print(tbl.to_string())